# 06b — Simulación Monte Carlo del Mundial 2026 **con D10Sformer**

Versión gemela del notebook 06 que **usa el Transformer fine-tuneado** (`finetune_weighted_15ep/best.pt`) en lugar del LogReg.

Diferencia clave vs `06_montecarlo_wc2026.ipynb`:

| | 06 (original) | 06b (este) |
|---|---|---|
| Predictor | `LogisticRegressionBaseline` re-entrenado | `D10Sformer` cargado de checkpoint |
| Features | tabla numérica (ELO, forma, h2h…) | secuencia tokenizada de `MatchDocument` |
| Inferencia | `logreg.predict_proba(x)` | `D10Sformer.forward(...)` → softmax sobre `result_logits` |

> ⚠️ Este notebook puede dar resultados *peores* que el 06 en términos de log-loss/Brier. Eso es esperado y **no es un bug** — es la diferencia real entre el Transformer y los baselines tabulares, que era una de las preguntas centrales del proyecto.

## 0 — Setup (Colab / local)

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, json, pickle, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# paths: ROOT ya definido en setup
SRC = ROOT / 'src'
PLAYGROUND_SERVICES = ROOT / 'playground' / 'services'

for p in (SRC, PLAYGROUND_SERVICES):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DATA_INTERIM = paths.data_interim
DATA_PROCESSED = paths.data_processed
VOCAB_PATH = paths.vocab_path
CKPT_PATH = ROOT / 'checkpoints' / 'finetune_weighted_15ep' / 'best.pt'

assert VOCAB_PATH.exists(), f'No encuentro vocab en {VOCAB_PATH}'
assert CKPT_PATH.exists(), f'No encuentro checkpoint en {CKPT_PATH}'
print(f'✓ vocab:    {VOCAB_PATH}')
print(f'✓ ckpt:     {CKPT_PATH}  ({CKPT_PATH.stat().st_size / 1e6:.1f} MB)')
print(f'✓ device:   {"cuda" if torch.cuda.is_available() else "cpu"}')

## 1 — Construir el corpus de features actuales (igual que 06)

In [ ]:
df_int = pd.read_parquet(DATA_INTERIM / 'international_matches_with_elo.parquet')
df_int['date'] = pd.to_datetime(df_int['date'])
print(f'Partidos internacionales: {len(df_int):,}')
print(f'Rango: {df_int.date.min().date()} → {df_int.date.max().date()}')

In [ ]:
# Recompute rolling features cronológicamente — idéntico a 06
from collections import defaultdict

df_sorted = df_int.sort_values('date').reset_index(drop=True).copy()
team_history = defaultdict(list)

form_pts_home, form_pts_away = [], []
recent_goals_home, recent_goals_away = [], []

def _roll_stats(history, k=5):
    last = history[-k:] if history else []
    if not last:
        return 1.0, 1.0
    pts = np.mean([h['pts'] for h in last])
    gfs = np.mean([h['gf'] for h in last])
    return pts, gfs

for _, row in df_sorted.iterrows():
    home, away = row['home_team'], row['away_team']
    hs, as_ = int(row['home_score']), int(row['away_score'])
    fph, gfh = _roll_stats(team_history[home])
    fpa, gfa = _roll_stats(team_history[away])
    form_pts_home.append(fph); recent_goals_home.append(gfh)
    form_pts_away.append(fpa); recent_goals_away.append(gfa)
    # actualizar después del partido
    if hs > as_: pts_h, pts_a = 3, 0
    elif hs < as_: pts_h, pts_a = 0, 3
    else: pts_h = pts_a = 1
    team_history[home].append({'pts': pts_h, 'gf': hs})
    team_history[away].append({'pts': pts_a, 'gf': as_})

df_sorted['home_form_pts'] = form_pts_home
df_sorted['away_form_pts'] = form_pts_away
df_sorted['home_recent_goals'] = recent_goals_home
df_sorted['away_recent_goals'] = recent_goals_away
print('✓ rolling features recomputadas')

In [ ]:
from simulation.bracket import WC2026_GROUPS, WC2026_GROUPS_RAW
WC_TEAMS = sorted({t for grp in WC2026_GROUPS.values() for t in grp})
print(f'Equipos del Mundial 2026: {len(WC_TEAMS)}')

def latest_state(team):
    rows = df_sorted[(df_sorted.home_team == team) | (df_sorted.away_team == team)]
    if rows.empty:
        return None
    last = rows.iloc[-1]
    is_home = last['home_team'] == team
    return {
        'elo': float(last['home_elo' if is_home else 'away_elo']),
        'form_pts': float(last['home_form_pts' if is_home else 'away_form_pts']),
        'recent_goals': float(last['home_recent_goals' if is_home else 'away_recent_goals']),
    }

team_features = {t: (latest_state(t) or {'elo': 1500.0, 'form_pts': 1.0, 'recent_goals': 1.0})
                 for t in WC_TEAMS}
tf_df = pd.DataFrame([{'team': t, **f} for t, f in team_features.items()]).sort_values('elo', ascending=False)
tf_df.head(10)

## 2 — Cargar el D10Sformer fine-tuneado

Acá está la diferencia clave con el notebook 06: en lugar de re-entrenar un LogReg, **cargamos los pesos del Transformer** y los envolvemos con `D10SformerPredictor`.

In [ ]:
# El módulo vive en playground/services/ (añadido al sys.path arriba)
try:
    from d10sformer_predictor import D10SformerPredictor
except ImportError:
    sys.path.insert(0, str(ROOT / 'playground'))
    from services.d10sformer_predictor import D10SformerPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time()
predictor_d10s = D10SformerPredictor(
    ckpt_path=CKPT_PATH,
    vocab_path=VOCAB_PATH,
    team_features=team_features,
    device=device,
)
print(f'✓ D10Sformer cargado en {time.time()-t0:.1f}s')
print()
for k, v in predictor_d10s.describe().items():
    print(f'  {k:<18} {v}')

## 3 — Test de humo: una predicción

In [ ]:
probs = predictor_d10s.predict('Argentina', 'France', venue='neutral')
print(f'Argentina vs France (neutral)')
print(f'  P(Argentina gana) = {probs[0]:.4f}')
print(f'  P(empate)         = {probs[1]:.4f}')
print(f'  P(Francia gana)   = {probs[2]:.4f}')
print(f'  suma              = {probs.sum():.4f}  (debe ser ~1.0)')
assert abs(probs.sum() - 1.0) < 1e-4, 'probs no suman 1'
assert probs.shape == (3,)

## 4 — Comparación rápida vs LogReg en partidos top

In [ ]:
# Cargar el LogReg del 06 para comparar predicción-a-predicción
from data.feature_engineering import build_feature_matrix, get_feature_columns
from models.baselines import LogisticRegressionBaseline

df_full = build_feature_matrix(
    df_sorted, windows_form=(5, 10), h2h_window=5, min_date='2014-01-01',
)
FEATURE_COLS = get_feature_columns(df_full)
X_full = df_full[FEATURE_COLS].values
y_full = df_full['result'].values
logreg = LogisticRegressionBaseline()
logreg.fit(X_full, y_full)

def _build_x(team_a, team_b, venue='neutral'):
    fa = team_features.get(team_a, {'elo': 1500, 'form_pts': 1.0, 'recent_goals': 1.0})
    fb = team_features.get(team_b, {'elo': 1500, 'form_pts': 1.0, 'recent_goals': 1.0})
    elo_diff = fa['elo'] - fb['elo']
    row = {
        'neutral': 1 if venue == 'neutral' else 0,
        'home_elo': fa['elo'], 'away_elo': fb['elo'], 'elo_diff': elo_diff,
        'expected_home_win_prob': 1 / (1 + 10 ** (-elo_diff / 400)),
        'home_rest_days': 7, 'away_rest_days': 7,
        'home_form5_pts': fa['form_pts'], 'home_form5_gf': fa['recent_goals'],
        'home_form5_ga': 1.0, 'home_form5_gd': 0.0, 'home_form5_n': 5,
        'away_form5_pts': fb['form_pts'], 'away_form5_gf': fb['recent_goals'],
        'away_form5_ga': 1.0, 'away_form5_gd': 0.0, 'away_form5_n': 5,
        'home_form10_pts': fa['form_pts'], 'home_form10_gf': fa['recent_goals'],
        'home_form10_ga': 1.0, 'home_form10_gd': 0.0, 'home_form10_n': 10,
        'away_form10_pts': fb['form_pts'], 'away_form10_gf': fb['recent_goals'],
        'away_form10_ga': 1.0, 'away_form10_gd': 0.0, 'away_form10_n': 10,
        'h2h_n_matches': 0, 'h2h_home_wins': 0, 'h2h_draws': 0, 'h2h_away_wins': 0,
        'h2h_avg_gd_for_home': 0.0,
    }
    x = pd.DataFrame([row])
    for col in FEATURE_COLS:
        if col not in x.columns:
            x[col] = 0.0
    return x[FEATURE_COLS].values

matchups = [
    ('Argentina', 'France'),
    ('Brazil', 'Spain'),
    ('Germany', 'Portugal'),
    ('Mexico', 'United States'),
    ('Saudi Arabia', 'Argentina'),  # famous WC22 upset
]

rows = []
for a, b in matchups:
    p_d10s = predictor_d10s.predict(a, b, 'neutral')
    p_lr   = logreg.predict_proba(_build_x(a, b, 'neutral'))[0]
    rows.append({
        'matchup': f'{a} vs {b}',
        'd10s_P(H)': p_d10s[0], 'd10s_P(D)': p_d10s[1], 'd10s_P(A)': p_d10s[2],
        'logreg_P(H)': p_lr[0], 'logreg_P(D)': p_lr[1], 'logreg_P(A)': p_lr[2],
    })
pd.DataFrame(rows).round(3)

## 5 — Monte Carlo del bracket con D10Sformer

In [ ]:
from simulation.simulator import PrecomputedPredictor, monte_carlo

# Pre-computamos los 48×47 pares dirigidos (misma idea que el 06 con LogReg)
precomputed = PrecomputedPredictor(
    predictor_d10s, WC_TEAMS, venue='neutral', verbose=True,
)

In [ ]:
N_SIMS = 10_000

# API real en src/simulation/simulator.py:
#   simulate_tournament(predictor, rng, fixed_results=None)
#   monte_carlo(predictor, n_iters, seed, ...) -> MonteCarloAggregation
t0 = time.time()
agg = monte_carlo(precomputed, n_iters=N_SIMS, seed=42, progress=True)
elapsed = time.time() - t0
print(f'✓ {N_SIMS} simulaciones en {elapsed:.1f}s ({N_SIMS/elapsed:.0f}/s)')

In [ ]:
df_pred = agg.to_dataframe()
df_pred['Equipo'] = df_pred['team']
df_pred = df_pred[['Equipo', 'P_group_advance', 'P_round_of_16', 'P_quarters', 'P_semis', 'P_final', 'P_champion']]
df_pred = df_pred.sort_values('P_champion', ascending=False).reset_index(drop=True)
print('=== Top 16 según D10Sformer ===')
print(df_pred.head(16).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

## 6 — Visualización: probabilidad de campeón

In [ ]:
TOP_N = 15
top = df_pred.head(TOP_N).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top['Equipo'], top['P_champion'] * 100, color='#D97706')
ax.set_xlabel('Probabilidad de campeón (%) — D10Sformer')
ax.set_title(f'Mundial 2026 — Top {TOP_N} según D10Sformer ({N_SIMS:,} sims)')
for i, (eq, p) in enumerate(zip(top['Equipo'], top['P_champion'])):
    ax.text(p * 100 + 0.1, i, f'{p*100:.1f}%', va='center', fontsize=10)
plt.tight_layout(); plt.show()

## 7 — Guardar resultados

In [ ]:
OUT = ROOT / 'reports' / 'wc2026_predictions_d10sformer.csv'
df_pred.to_csv(OUT, index=False)
print(f'✓ guardado en {OUT}')

## 8 — Validaciones de coherencia del simulador (tabla resumen)

Mismas comprobaciones que en el notebook 06 (LogReg), pero con el predictor D10Sformer.

In [ ]:
from scipy.stats import spearmanr

STAGES = ['P_group_advance', 'P_round_of_16', 'P_quarters', 'P_semis', 'P_final', 'P_champion']

sum_champ = df_pred['P_champion'].sum()
mono_ok = all(
    all(row[STAGES[i]] >= row[STAGES[i + 1]] - 1e-12 for i in range(len(STAGES) - 1))
    for _, row in df_pred.iterrows()
)

df_elo = df_pred.merge(tf_df[['team', 'elo']], left_on='Equipo', right_on='team')
rho, pval = spearmanr(df_elo['elo'], df_elo['P_champion'])

print('| Métrica | Resultado |')
print('|---|---|')
print(f'| Σ P(campeón) sobre 48 equipos | {sum_champ:.3f} (±1e-9) |')
print(f'| Monotonicidad | {"P(grupo) ≥ P(16avos) ≥ … ≥ P(campeón) ✓" if mono_ok else "FALLA — revisar agregación"} |')
print(f'| Spearman ELO ↔ P(campeón) | ρ = {rho:.3f} (p = {pval:.2e}) |')
print(f'| Campeón más probable | {df_pred.iloc[0]["Equipo"]} ({df_pred.iloc[0]["P_champion"]*100:.2f}%) |')

In [ ]:
# Final más frecuente: requiere registrar champion + runner_up (no está en el CSV agregado)
N_DEEP = 2_000
rng_deep = np.random.default_rng(123)
final_matchups = Counter()

for _ in range(N_DEEP):
    res = simulate_tournament(precomputed, rng_deep)
    if res.champion and res.runner_up:
        pair = tuple(sorted([res.champion, res.runner_up]))
        final_matchups[pair] += 1

top_pair, top_count = final_matchups.most_common(1)[0]
pct = 100 * top_count / N_DEEP
print(f'| Final más frecuente | {top_pair[0]} vs {top_pair[1]} (~{pct:.1f}% de {N_DEEP:,} sims) |')
print('\nTop 10 finales:')
for pair, count in final_matchups.most_common(10):
    print(f'  {pair[0]:<18} vs {pair[1]:<18}  {count:4d}  ({100*count/N_DEEP:.2f}%)')